# MethylSeg test workspace

This historical notebook runs local MethylSeg parameter sweeps for the chromatin samples `TE5.wgbs` and `ESO26.wgbs`. It is retained for provenance and may require the original MethylSeg and legacy helper versions to execute. Use `methylseg_sweep_figure_browser.ipynb` to inspect the archived results.

In this archive, generated outputs are rooted at `results/00_methylseg_development/`, including:

- comparator-shaped MethylSeg outputs under `results/methylseg/...`
- MethylSeg-only chromatin and LAD fallback analyses under `results/functional_analysis/...`
- local synthetic prep, MethylSeg runs, and synthetic summaries under `results/synthetic/...`

When tracked MethylSeg config values change, stale `out/` products are automatically invalidated while `prep/` artifacts are preserved.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from methylsegtest_utils import (
    DEFAULT_SWEEP_ROOT,
    DEFAULT_SYNTHETIC_SAMPLE_IDS,
    default_config,
    generate_random_search_config_variants,
    load_sweep_result_tables,
    run_methylseg_config_sweep,
    sample_manifest_dataframe,
)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)


In [2]:
base_config = default_config()

base_config["samples"] = ["TE5.wgbs", "ESO26.wgbs"]
base_config["wgbs_hmm_type"] = "sticky"
base_config["wgbs_hmm_params"] = {}
base_config["hm450_hmm_type"] = "ct"
base_config["hm450_hmm_params"] = {
    "n_emissions": 4,
    "holding_time_guess": 300_000,
    "algorithm": "forward-backward",
    "max_iter": 25,
    "tol": 1e-2,
}
base_config["min_coverage"] = 10
base_config["n_states"] = 4
base_config["int_low_cutoff"] = 0.2
base_config["int_high_cutoff"] = 0.7
base_config["high_cutoff"] = 0.7
base_config["random_state"] = 42
base_config["max_workers"] = 4
base_config["run_chromatin"] = True
base_config["run_chromatin_deeptools"] = True
base_config["chromatin_include_heatmaps"] = False
base_config["run_lad"] = True
base_config["run_synthetic_prep"] = True
base_config["run_synthetic_aggregate"] = True
base_config["synthetic_overwrite"] = False
base_config["synthetic_sample_ids"] = list(DEFAULT_SYNTHETIC_SAMPLE_IDS)
base_config["clear_out_dirs_before_run"] = True
base_config["force_recreate"] = False
base_config["print_logs"] = False

sweep_name = "random_parallel_sweep"
sweep_root = DEFAULT_SWEEP_ROOT / sweep_name
max_config_workers = 25

random_search_seed = 42
random_search_n_configs = 25
random_search_prefix = "random"

config_variants = generate_random_search_config_variants(
    n_configs=random_search_n_configs,
    seed=random_search_seed,
    config_name_prefix=random_search_prefix,
)

configs_to_run = config_variants

{
    "sweep_root": str(Path(sweep_root).resolve()),
    "max_config_workers": max_config_workers,
    "random_search_seed": random_search_seed,
    "random_search_n_configs": random_search_n_configs,
    "n_configs": len(configs_to_run),
    "config_names": [config["config_name"] for config in configs_to_run[:5]],
}


{'sweep_root': 'results/00_methylseg_development/random_parallel_sweep',
 'max_config_workers': 25,
 'random_search_seed': 42,
 'random_search_n_configs': 25,
 'n_configs': 25,
 'config_names': ['random_01',
  'random_02',
  'random_03',
  'random_04',
  'random_05']}

In [3]:
def _merge_preview(base, overrides):
    merged = dict(base)
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = {**merged[key], **value}
        else:
            merged[key] = value
    return merged

config_plan_rows = []
for variant in configs_to_run:
    merged = _merge_preview(base_config, variant)
    config_plan_rows.append(
        {
            "config_name": merged["config_name"],
            "out_root": str((Path(sweep_root) / merged["config_name"] / "results").resolve()),
            "samples": ",".join(merged["samples"]),
            "max_workers": merged["max_workers"],
            "clean_min_cpgs": merged["clean_min_cpgs"],
            "clean_min_region_length": merged["clean_min_region_length"],
            "clean_merge_gap_bp": merged["clean_merge_gap_bp"],
            "wgbs_window_specs": str(merged["wgbs_window_specs"]),
            "hm450_window_specs": str(merged["hm450_window_specs"]),
            "hm450_holding_time_guess": merged["hm450_hmm_params"]["holding_time_guess"],
            "run_chromatin": merged["run_chromatin"],
            "run_lad": merged["run_lad"],
            "run_synthetic_aggregate": merged["run_synthetic_aggregate"],
        }
    )

config_plan_df = pd.DataFrame(config_plan_rows).sort_values("config_name").reset_index(drop=True)
display(sample_manifest_dataframe(base_config["samples"]))
display(config_plan_df)
print(f"Sweep root: {Path(sweep_root).resolve()}")
print(
    f"Random search uses seed={random_search_seed} to generate {len(configs_to_run)} configs with 1-4 window sizes between 5kb and 2MB."
)
print("Each config gets its own results subtree, so config-specific cleanup only touches that config's outputs.")


,sample,sample_id,genome,meth_file
0,ESO26.wgbs,ESO26,hg38,data/methylation_data/ESO26.wgbs.bed.gz
1,TE5.wgbs,TE5,hg38,data/methylation_data/TE5.wgbs.bed.gz


,config_name,out_root,samples,max_workers,clean_min_cpgs,clean_min_region_length,clean_merge_gap_bp,wgbs_window_specs,hm450_window_specs,hm450_holding_time_guess,run_chromatin,run_lad,run_synthetic_aggregate
0,random_01,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,43,171720,25783,"[(69336, '69.34kb')]","[(69336, '69.34kb')]",1547935,True,True,True
1,random_02,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,79,102645,38434,"[(8791, '8.79kb'), (478081, '478.08kb'), (1728215, '1.73mb')]","[(8791, '8.79kb'), (478081, '478.08kb'), (1728215, '1.73mb')]",1434983,True,True,True
2,random_03,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,55,88683,135138,"[(46111, '46.11kb'), (236780, '236.78kb'), (691584, '691.58kb'), (1289639, '1.29mb')]","[(46111, '46.11kb'), (236780, '236.78kb'), (691584, '691.58kb'), (1289639, '1.29mb')]",900827,True,True,True
3,random_04,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,6,171659,248290,"[(138686, '138.69kb')]","[(138686, '138.69kb')]",1775791,True,True,True
4,random_05,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,6,194140,133706,"[(41828, '41.83kb'), (469418, '469.42kb')]","[(41828, '41.83kb'), (469418, '469.42kb')]",1263366,True,True,True
5,random_06,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,15,148675,204915,"[(6501, '6.5kb'), (16048, '16.05kb'), (81923, '81.92kb'), (530116, '530.12kb')]","[(6501, '6.5kb'), (16048, '16.05kb'), (81923, '81.92kb'), (530116, '530.12kb')]",1093184,True,True,True
6,random_07,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,80,37894,138882,"[(35220, '35.22kb'), (46018, '46.02kb'), (83326, '83.33kb'), (1646222, '1.65mb')]","[(35220, '35.22kb'), (46018, '46.02kb'), (83326, '83.33kb'), (1646222, '1.65mb')]",1489550,True,True,True
7,random_08,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,22,112900,200944,"[(86454, '86.45kb')]","[(86454, '86.45kb')]",660459,True,True,True
8,random_09,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,43,160953,252508,"[(32491, '32.49kb'), (331972, '331.97kb'), (732082, '732.08kb'), (733920, '733.92kb')]","[(32491, '32.49kb'), (331972, '331.97kb'), (732082, '732.08kb'), (733920, '733.92kb')]",874360,True,True,True
9,random_10,results/00_methylseg_development...,"TE5.wgbs,ESO26.wgbs",4,14,166548,59972,"[(28133, '28.13kb'), (298445, '298.44kb')]","[(28133, '28.13kb'), (298445, '298.44kb')]",1273560,True,True,True


Sweep root: results/00_methylseg_development/random_parallel_sweep
Random search uses seed=42 to generate 25 configs with 1-4 window sizes between 5kb and 2MB.
Each config gets its own results subtree, so config-specific cleanup only touches that config's outputs.


## Run MethylSeg on TE5 and ESO26

This uses the existing comparator-side prep logic and writes comparator-compatible MethylSeg outputs under `results/methylseg/<sample>/...`.

In [ ]:
sweep_results = run_methylseg_config_sweep(
    configs_to_run,
    base_config=base_config,
    sweep_root=sweep_root,
    max_config_workers=max_config_workers,
)

sweep_manifest_df = sweep_results["manifest_df"]
sweep_results_df = sweep_results["results_df"]
config_lookup = {
    config["config_name"]: config
    for config in sweep_results["normalized_configs"]
}
config_tables = load_sweep_result_tables(sweep_results_df=sweep_results_df)

display(sweep_manifest_df)
display(sweep_results_df[[
    "config_name",
    "status",
    "out_root",
    "n_primary_samples",
    "n_chromatin_rows",
    "n_lad_rows",
    "n_synthetic_runs",
    "n_synthetic_summary_rows",
    "error",
]])

for row in sweep_results_df.itertuples(index=False):
    display(Markdown(f"### {row.config_name} primary MethylSeg outputs"))
    tables = config_tables[row.config_name]
    if not tables["methylseg_manifest_df"].empty:
        display(tables["methylseg_manifest_df"])
    if not tables["methylseg_summary_df"].empty:
        display(tables["methylseg_summary_df"])
    if row.status != "success" and row.traceback:
        print(row.traceback)


[methylsegtests] Running 25 config(s) with up to 25 config worker(s).
[methylsegtests] Starting config run: random_01[methylsegtests] Starting config run: random_02[methylsegtests] Starting config run: random_03[methylsegtests] Starting config run: random_05[methylsegtests] Starting config run: random_04[methylsegtests] Starting config run: random_08[methylsegtests] Starting config run: random_09
[methylsegtests] Starting config run: random_06[methylsegtests] Starting config run: random_10
[methylsegtests] Starting config run: random_07
[methylsegtests] Starting config run: random_11[methylsegtests] Starting config run: random_12
[methylsegtests] Starting config run: random_13
[methylsegtests] Starting config run: random_18[methylsegtests] Starting config run: random_17
[methylsegtests] Starting config run: random_16
[methylsegtests] Starting config run: random_19[methylsegtests] Starting config run: random_20[methylsegtests] Starting config run: random_22
[methylsegtests] Starting con

One of the groups defined in the bed file is too small.
Groups that are too small can't be plotted. 


$ plotProfile --numPlotsPerRow 3 -m results/00_methylseg_development/random_parallel_sweep/random_12/results/functional_analysis/chromatin_fallback/deeptools/TE5.wgbs/TE5.wgbs.h3k36me2.matrix.gz --perGroup --samplesLabel TE5 --regionsLabel 'MethylSeg WGBS' 'MethylSeg HM450K' --startLabel 'Region start' --endLabel 'Region end' --plotTitle 'TE5.wgbs H3K36me2 profile across tool-derived regions' -out results/00_methylseg_development/random_parallel_sweep/random_12/results/functional_analysis/chromatin_fallback/deeptools/TE5.wgbs/TE5.wgbs.h3k36me2.profile.png
$ computeMatrix scale-regions -p 1 -S data/chromatin_data/ESO26.h3k36me2.bw -R results/00_methylseg_development/random_parallel_sweep/random_12/results/functional_analysis/chromatin_fallback/deeptools/ESO26.wgbs/ESO26.wgbs.methylseg.deeptools_regions.bed results/00_methylseg_development/random_parallel_sweep/random_12/results/functional_analysis/chromatin_fallback/deeptools/ESO26.wgbs/ESO26.wgbs.methylseg_hm450k.deeptools_regions.bed 

$ plotProfile --numPlotsPerRow 3 -m results/00_methylseg_development/random_parallel_sweep/random_06/results/functional_analysis/chromatin_fallback/deeptools/TE5.wgbs/TE5.wgbs.h3k36me2.matrix.gz --perGroup --samplesLabel TE5 --regionsLabel 'MethylSeg WGBS' 'MethylSeg HM450K' --startLabel 'Region start' --endLabel 'Region end' --plotTitle 'TE5.wgbs H3K36me2 profile across tool-derived regions' -out results/00_methylseg_development/random_parallel_sweep/random_06/results/functional_analysis/chromatin_fallback/deeptools/TE5.wgbs/TE5.wgbs.h3k36me2.profile.png
$ computeMatrix scale-regions -p 1 -S data/chromatin_data/ESO26.h3k36me2.bw -R results/00_methylseg_development/random_parallel_sweep/random_06/results/functional_analysis/chromatin_fallback/deeptools/ESO26.wgbs/ESO26.wgbs.methylseg.deeptools_regions.bed results/00_methylseg_development/random_parallel_sweep/random_06/results/functional_analysis/chromatin_fallback/deeptools/ESO26.wgbs/ESO26.wgbs.methylseg_hm450k.deeptools_regions.bed 

Registered S3 methods overwritten by 'GenomeInfoDb':
  method                from   
  as.data.frame.Seqinfo Seqinfo
  merge.Seqinfo         Seqinfo
  summary.Seqinfo       Seqinfo
  update.Seqinfo        Seqinfo
There were 26 warnings (use warnings() to see them)
Warning messages:
1: package ‘GenomicRanges’ was built under R version 4.5.2 
2: package ‘BiocGenerics’ was built under R version 4.5.2 
3: package ‘generics’ was built under R version 4.5.2 
4: package ‘S4Vectors’ was built under R version 4.5.2 
5: package ‘IRanges’ was built under R version 4.5.2 
6: package ‘Seqinfo’ was built under R version 4.5.2 
Registered S3 methods overwritten by 'GenomeInfoDb':
  method                from   
  as.data.frame.Seqinfo Seqinfo
  merge.Seqinfo         Seqinfo
  summary.Seqinfo       Seqinfo
  update.Seqinfo        Seqinfo
There were 26 warnings (use warnings() to see them)
Warning messages:
1: package ‘GenomicRanges’ was built under R version 4.5.2 
2: package ‘BiocGenerics’ was built 

...

[methylsegtests] Reusing existing synthetic samples and configs; no synthetic dataprep rerun was needed.
[methylsegtests] Cleared 1 cached output path(s) for synthetic methylseg results before rerun.
[methylsegtests] Running 3 synthetic sample(s) with up to 3 worker(s).
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_1_hg38
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_2_hg38
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_3_hg38
[methylsegtests] Reusing existing synthetic samples and configs; no synthetic dataprep rerun was needed.
[methylsegtests] Cleared 1 cached output path(s) for synthetic methylseg results before rerun.
[methylsegtests] Running 3 synthetic sample(s) with up to 3 worker(s).
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_2_hg38[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_3_hg38

[me

Registered S3 methods overwritten by 'GenomeInfoDb':
  method                from   
  as.data.frame.Seqinfo Seqinfo
  merge.Seqinfo         Seqinfo
  summary.Seqinfo       Seqinfo
  update.Seqinfo        Seqinfo
There were 26 warnings (use warnings() to see them)
Warning messages:
1: package ‘GenomicRanges’ was built under R version 4.5.2 
2: package ‘BiocGenerics’ was built under R version 4.5.2 
3: package ‘generics’ was built under R version 4.5.2 
4: package ‘S4Vectors’ was built under R version 4.5.2 
5: package ‘IRanges’ was built under R version 4.5.2 
6: package ‘Seqinfo’ was built under R version 4.5.2 
Registered S3 methods overwritten by 'GenomeInfoDb':
  method                from   
  as.data.frame.Seqinfo Seqinfo
  merge.Seqinfo         Seqinfo
  summary.Seqinfo       Seqinfo
  update.Seqinfo        Seqinfo
There were 26 warnings (use warnings() to see them)
Warning messages:
1: package ‘GenomicRanges’ was built under R version 4.5.2 
2: package ‘BiocGenerics’ was built 

[methylsegtests] Reusing existing synthetic samples and configs; no synthetic dataprep rerun was needed.
[methylsegtests] Cleared 1 cached output path(s) for synthetic methylseg results before rerun.
[methylsegtests] Running 3 synthetic sample(s) with up to 3 worker(s).
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_1_hg38
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_2_hg38
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_3_hg38
[methylsegtests] Reusing existing synthetic samples and configs; no synthetic dataprep rerun was needed.
[methylsegtests] Cleared 1 cached output path(s) for synthetic methylseg results before rerun.
[methylsegtests] Running 3 synthetic sample(s) with up to 3 worker(s).
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_3_hg38[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_1_hg38[meth

Registered S3 methods overwritten by 'GenomeInfoDb':
  method                from   
  as.data.frame.Seqinfo Seqinfo
  merge.Seqinfo         Seqinfo
  summary.Seqinfo       Seqinfo
  update.Seqinfo        Seqinfo
There were 26 warnings (use warnings() to see them)
Warning messages:
1: package ‘GenomicRanges’ was built under R version 4.5.2 
2: package ‘BiocGenerics’ was built under R version 4.5.2 
3: package ‘generics’ was built under R version 4.5.2 
4: package ‘S4Vectors’ was built under R version 4.5.2 
5: package ‘IRanges’ was built under R version 4.5.2 
6: package ‘Seqinfo’ was built under R version 4.5.2 
Registered S3 methods overwritten by 'GenomeInfoDb':
  method                from   
  as.data.frame.Seqinfo Seqinfo
  merge.Seqinfo         Seqinfo
  summary.Seqinfo       Seqinfo
  update.Seqinfo        Seqinfo
There were 26 warnings (use warnings() to see them)
Warning messages:
1: package ‘GenomicRanges’ was built under R version 4.5.2 
2: package ‘BiocGenerics’ was built 

[methylsegtests] Reusing existing synthetic samples and configs; no synthetic dataprep rerun was needed.
[methylsegtests] Cleared 1 cached output path(s) for synthetic methylseg results before rerun.
[methylsegtests] Running 3 synthetic sample(s) with up to 3 worker(s).
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_1_hg38[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_2_hg38

[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_3_hg38
[methylsegtests] Reusing existing synthetic samples and configs; no synthetic dataprep rerun was needed.
[methylsegtests] Cleared 1 cached output path(s) for synthetic methylseg results before rerun.
[methylsegtests] Running 3 synthetic sample(s) with up to 3 worker(s).
[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_2_hg38[methylsegtests] Starting MethylSeg sample run: synthetic_WGBS_colon_primary_normal_1_hg38[meth

## Chromatin fallback analysis

This is a notebook-local, MethylSeg-only fallback summary for `TE5.wgbs` and `ESO26.wgbs`. It does not call the full multi-tool `run_chromatin.py` pipeline, but it can generate deepTools profile plots for the two MethylSeg region sets.

In [ ]:
for row in sweep_results_df.itertuples(index=False):
    display(Markdown(f"### {row.config_name} chromatin results"))
    tables = config_tables[row.config_name]
    chromatin_metrics_df = tables["chromatin_metrics_df"]
    deeptools_outputs_df = tables["chromatin_deeptools_outputs_df"]
    config_for_row = config_lookup[row.config_name]

    if chromatin_metrics_df.empty:
        print("Chromatin fallback analysis skipped or no outputs were produced for this config.")
        continue

    display(chromatin_metrics_df)
    if not deeptools_outputs_df.empty:
        display(deeptools_outputs_df)
        for deeptools_row in deeptools_outputs_df.itertuples(index=False):
            print(f"deepTools profile for {row.config_name} / {deeptools_row.sample}: {deeptools_row.profile_path}")
            display(Image(filename=deeptools_row.profile_path))
            heatmap_path = getattr(deeptools_row, "heatmap_path", "")
            if config_for_row["chromatin_include_heatmaps"] and isinstance(heatmap_path, str) and heatmap_path:
                print(f"deepTools heatmap for {row.config_name} / {deeptools_row.sample}: {heatmap_path}")
                display(Image(filename=heatmap_path))


## LAD fallback analysis

This is a notebook-local, MethylSeg-only fallback summary for LAD overlap. It prepares its own local reference assets under `results/functional_analysis/lad_fallback/`.

In [ ]:
for row in sweep_results_df.itertuples(index=False):
    display(Markdown(f"### {row.config_name} LAD results"))
    lad_metrics_df = config_tables[row.config_name]["lad_metrics_df"]
    if lad_metrics_df.empty:
        print("LAD fallback analysis skipped or no outputs were produced for this config.")
        continue
    display(lad_metrics_df)


## Synthetic prep and MethylSeg-only synthetic summary

When enabled, this section keeps synthetic artifacts under `results/synthetic/`, reuses local prep if present, runs MethylSeg on the selected synthetic samples, and produces a local MethylSeg-only synthetic overlap summary.

In [ ]:
for row in sweep_results_df.itertuples(index=False):
    display(Markdown(f"### {row.config_name} synthetic results"))
    tables = config_tables[row.config_name]

    if not tables["synthetic_manifest_df"].empty:
        display(tables["synthetic_manifest_df"])
    else:
        print("Synthetic prep skipped or no synthetic manifest is available for this config.")

    if not tables["synthetic_methylseg_manifest_df"].empty:
        display(tables["synthetic_methylseg_manifest_df"])
    else:
        print("Synthetic MethylSeg execution skipped or no synthetic run manifest is available for this config.")

    if not tables["synthetic_summary_df"].empty:
        display(tables["synthetic_summary_df"])
    else:
        print("Synthetic summary skipped or no outputs were produced for this config.")


## Notes

- Edit `random_search_seed` to regenerate a reproducible random sweep.
- Edit `random_search_n_configs` to change how many configs are generated; the default is 25.
- The random generator samples 1-4 window sizes between 5kb and 2MB, `holding_time_guess` between 100 and 2,000,000, `clean_min_cpgs` between 0 and 100, `clean_min_region_length` between 0 and 200,000, and `clean_merge_gap_bp` between 0 and 300,000.
- Each config runs in `results/00_methylseg_development/<sweep_name>/<config_name>/results`, so outputs and cleanup stay isolated.
- `max_config_workers` controls how many configs run at once; each config also uses its own `max_workers` for sample-level parallelism.
- With `clear_out_dirs_before_run=True`, each rerun clears cached MethylSeg `out/` directories before execution while preserving `prep/` artifacts.
- Tracked config changes still invalidate stale `out/` products, but only inside that config's results subtree when forced clearing is disabled.
